# Cross-sell pipeline — end-to-end walkthrough

An interactive tour of every analytic component in this framework, one stage at
a time, with the real inputs and outputs of each step on screen so you can poke
at them:

| Stage | Package | Components covered |
|---|---|---|
| **1. Preprocessing** | `nbo_data_preprocessing` | `load_data` → `split_outcome_rows` → `compute_anchors` (both designs) → percentile grids → `build_features` → flat / hazard datasets → exclusions |
| **2. Modelling** | `nbo_data_modeling` | `load_data` (downsampling) → column typing → pipeline → group cross-validation → hazard→customer scores → metrics / calibration / coefficients |
| **3. Reporting** | `nbo_report` | Markov sequence profiling → lineage / IV / anchor-asymmetry → deciles → refit + reason codes → segments |

**How to use it:** set the parameters in the next cell, then *Run All*. Every
cell re-derives from the one before it, so you can edit any intermediate frame
and re-run downstream cells to see the effect. Expect roughly **2–4 minutes**
end to end at the default size (most of it in `build_features` and
cross-validation).

**Your own data:** by default the notebook generates a synthetic extract with
`make_synthetic.py`. To use your own file instead, set `USER_DATA_PATH` below.
The file must match the raw schema your `nbo_data_preprocessing/config.yaml`
describes — the column *roles* (`columns:`) and every key under
`numeric_columns:` must exist in the file. Nothing in this notebook hardcodes a
column name; everything is read from the configs, so pointing the configs at a
different dataset re-points the whole walkthrough.

Everything the walkthrough writes lands under `outputs/walkthrough/`, so it
never touches the artifacts of a real `eda.py` / `main.py` run.

## Parameters

| Parameter | Meaning |
|---|---|
| `USER_DATA_PATH` | `None` → generate synthetic data. Otherwise an absolute or repo-relative path to a raw extract matching the preprocessing config's schema. |
| `WALKTHROUGH_CUSTOMERS` | Size of the generated synthetic file (ignored when `USER_DATA_PATH` is set). Kept small so the notebook stays interactive. |
| `FORCE_REGENERATE` | Re-generate the synthetic file even if one already exists from a previous run. |
| `RUN_CV` | Run cross-validation inside the notebook. Turning it off skips the modelling and targeting sections. |
| `CV_REPEATS` | CV repeats for the in-notebook run (the real config ships with more; one repeat keeps the loop fast). |
| `TOP_N_PREVIEW` | How many top-scored customers to show with reason codes. |

In [ ]:
# ---------------------------------------------------------------------
# Parameters — edit these, then Run All.
# ---------------------------------------------------------------------

# Path to YOUR raw extract, or None to generate synthetic data.
# e.g. USER_DATA_PATH = r"C:/data/my_raw_acquisitions.csv"
USER_DATA_PATH = None

WALKTHROUGH_CUSTOMERS = 4000
FORCE_REGENERATE = False

RUN_CV = True
CV_REPEATS = 1

TOP_N_PREVIEW = 10

In [ ]:
# ---------------------------------------------------------------------
# Setup: import the pipeline exactly as the entry points do.
# ---------------------------------------------------------------------
import copy
import json
import os
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import HTML, display

# The repo root is wherever this notebook lives.
NB_ROOT = os.path.abspath(os.getcwd())
if NB_ROOT not in sys.path:
    sys.path.insert(0, NB_ROOT)

from nbo_report import charts, config, html, profiling, sequences, targeting

ctx = config.ReportContext("report_config.yaml")
pre = config.preprocess_module(ctx)      # nbo_data_preprocessing/preprocess.py
mdl = config.model_module(ctx)           # nbo_data_modeling/model.py

# Column ROLES, straight from the configs — the notebook never hardcodes
# a dataset column name.
COL = ctx.pre["columns"]                 # raw-file roles (id, date, target, ...)
SCH = ctx.schema                         # engineered-output names
SCH_A = SCH["anchors"]                   # columns of compute_anchors' frame
ID, DATE, TARGET = COL["id"], COL["date"], COL["target"]
CUST = SCH["customer"]

# Everything this notebook writes goes here, away from real runs.
WALK = os.path.join(config.ROOT, "outputs", "walkthrough")
for sub in ("", "model_data", "artifacts", "model_output"):
    os.makedirs(os.path.join(WALK, sub), exist_ok=True)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: "%.4f" % v)

def show(chart_html):
    """Render a charts.py <img> (or any HTML fragment) inline."""
    display(HTML(chart_html))

# A scoped slice of the report stylesheet so rendered report cards look
# right inside the notebook without restyling the notebook itself.
NB_CSS = """<style>
.nbo{font-family:system-ui,sans-serif;font-size:13px;max-width:1100px}
.nbo .card{border:1px solid #e1e0d9;border-radius:10px;padding:16px 18px;margin:8px 0;background:#fcfcfb;color:#0b0b0b}
.nbo table{border-collapse:collapse;font-size:12.5px}
.nbo thead th{text-align:left;border-bottom:1.5px solid #c3c2b7;padding:6px 12px 6px 0;color:#52514e}
.nbo tbody td{padding:5px 12px 5px 0;border-bottom:1px solid #e1e0d9;white-space:nowrap}
.nbo .callout{border-left:3px solid #2a78d6;padding:10px 14px;margin:8px 0;background:#fcfcfb;border-top:1px solid #e1e0d9;border-right:1px solid #e1e0d9;border-bottom:1px solid #e1e0d9;border-radius:0 8px 8px 0}
.nbo .callout .t{font-weight:600;display:block}
.nbo .callout .b{color:#52514e}
.nbo .badge{font-size:11px;font-weight:600;padding:1px 8px;border-radius:999px;border:1px solid #e1e0d9}
.nbo h3{font-size:14px;margin:14px 0 6px}
.nbo p.empty{color:#52514e;border:1px dashed #c3c2b7;border-radius:8px;padding:10px 12px}
.nbo .chart-note{color:#898781;font-size:12px}
</style>"""

def show_card(card_html):
    display(HTML(NB_CSS + "<div class='nbo'>" + card_html + "</div>"))

print("repo root :", config.ROOT)
print("outputs   :", WALK)
print("roles     : id=%r  date=%r  target=%r" % (ID, DATE, TARGET))
print("python %s | pandas %s | numpy %s"
      % (sys.version.split()[0], pd.__version__, np.__version__))

---
## Stage 0 — the raw input

One row per **acquisition** (a product an entity opened), plus one **outcome
row** per entity that had the target event. Outcome rows carry only the id, the
date, and a non-null value in the target column — every other column is null.
That shape is the contract everything downstream depends on.

In [ ]:
# Resolve the input: your file, or a generated synthetic one.
if USER_DATA_PATH:
    raw_path = os.path.abspath(USER_DATA_PATH)
    if not os.path.exists(raw_path):
        raise FileNotFoundError("USER_DATA_PATH does not exist: %s" % raw_path)
    print("using YOUR data:", raw_path)
else:
    raw_path = os.path.join(WALK, "raw_synthetic.csv")
    if FORCE_REGENERATE or not os.path.exists(raw_path):
        config.run_script(
            ctx.cfg["io"]["synthetic_script"],
            ["--customers", WALKTHROUGH_CUSTOMERS, "--out", raw_path,
             "--seed", ctx.cfg["synthetic"]["seed"],
             "--base-rate", ctx.cfg["synthetic"]["base_rate"]],
            label="make_synthetic.py")
    print("using synthetic data:", raw_path)

# The walkthrough's own copy of the preprocessing config: identical to
# the real one except that io: points at the walkthrough sandbox. This
# is the same trick that lets the pipeline move between datasets —
# behaviour lives in config, not code.
cfg_pre = copy.deepcopy(ctx.pre)
cfg_pre["io"] = {
    "input_path":   raw_path,
    "output_dir":   os.path.join(WALK, "model_data"),
    "artifact_dir": os.path.join(WALK, "artifacts"),
}

# Validate the schema BEFORE doing anything, with the pipeline's own
# check — most useful when USER_DATA_PATH points at your file.
_sample = pd.read_csv(raw_path, nrows=50)
pre.require_columns(_sample, [ID, DATE, TARGET, COL["product_type"],
                              COL["tiebreak"]], "columns")
pre.require_columns(_sample, list(cfg_pre["numeric_columns"]), "numeric_columns")
print("schema OK — all configured columns present")

In [ ]:
raw = pd.read_csv(raw_path)
is_outcome = raw[TARGET].notna()
print("%s rows | %s entities | %s outcome rows"
      % (len(raw), raw[ID].nunique(), int(is_outcome.sum())))

print("\nan acquisition row:")
display(raw[~is_outcome].head(3))
print("an outcome row — note every column but id/date/target is null:")
display(raw[is_outcome].head(3))

print("nulls per column (acquisition rows only):")
display(raw[~is_outcome].isna().mean().rename("null share").to_frame().T)

---
## Stage 1 — preprocessing (`nbo_data_preprocessing/preprocess.py`)

```
load → split outcome rows → pick anchor → build features → attach labels
```

Two invariants rule this stage (see `CLAUDE.md`):

1. **Outcome rows are split away before any aggregation** — a groupby that
   still sees them silently corrupts counts and last-row lookups.
2. **No feature may use a row dated after its anchor**, or encode the distance
   from the anchor to the outcome / extract date. Both leak the label.

In [ ]:
# --- 1a. load_data: parses dates, sorts stably, derives the family rollup ---
df = pre.load_data(cfg_pre)

fam_col = SCH["family"]
if fam_col in df.columns:
    print("derived %r from the product_families map in config.yaml:" % fam_col)
    display(df[df[TARGET].isna()]
            .groupby(fam_col)[COL["product_type"]]
            .agg(rows="size", product_types="nunique")
            .sort_values("rows", ascending=False))
else:
    print("no family rollup column (%r) in this dataset" % fam_col)

In [ ]:
# --- 1b. split_outcome_rows: THE first invariant, applied ---
acq, out = pre.split_outcome_rows(df, cfg_pre)
print("acquisitions: %6d rows, %5d entities" % (len(acq), acq[ID].nunique()))
print("outcomes:     %6d rows, %5d entities" % (len(out), out[ID].nunique()))
assert acq[TARGET].notna().sum() == 0, "an outcome row leaked into acq!"
print("\ncheck passed: no outcome row survives in the acquisition frame")

### 1c. Anchors — where each entity's history is cut

The **anchor** is the moment the model pretends "today" is. Only rows at or
before it are visible. The two designs pick it differently, and that single
difference is what separates the two output datasets:

- **fixed** (`flat_dataset.csv`): non-converters anchor at their last
  acquisition at least `horizon_months` before the extract date, so both
  classes get equal time at risk. Converters whose outcome falls beyond the
  horizon are **excluded, never relabelled 0**.
- **hazard** (`hazard_dataset.csv`): everyone anchors at their last
  acquisition; censoring is handled later by interval expansion, so nobody is
  dropped for timing.

Run both and compare who each rule keeps and why it drops the rest.

In [ ]:
anch_f = pre.compute_anchors(acq, out, cfg_pre, mode="fixed")
anch_h = pre.compute_anchors(acq, out, cfg_pre, mode="hazard")

KEEP, REASON, EVENT = SCH_A["keep"], "reason", SCH_A["event"]
cmp = pd.concat([
    anch_f.groupby([KEEP, REASON]).size().rename("fixed"),
    anch_h.groupby([KEEP, REASON]).size().rename("hazard"),
], axis=1).fillna(0).astype(int)
print("who each anchor rule keeps (and why it drops the rest):")
display(cmp)

kept_f = anch_f[anch_f[KEEP]]
print("\nfixed design: %d kept, of which %d converters (%.2f%%)"
      % (len(kept_f), int(kept_f[EVENT].sum()),
         100 * kept_f[EVENT].mean()))

In [ ]:
# --- trace one entity through the whole pipeline ---------------------
# Pick a converter with a decent history; every later stage shows this
# same entity so you can follow one thread end to end. Override
# trace_id with any id from your data to trace a different one.
_kept = anch_h[(anch_h[KEEP]) & (anch_h[EVENT] == 1)]
_n_by_id = acq.groupby(ID).size()
trace_id = (_kept.set_index(SCH_A["customer"]).index
            .to_series().map(_n_by_id).sort_values(ascending=False).index[0])

print("tracing entity:", trace_id)
print("\ntheir raw rows (acquisitions + outcome):")
display(df[df[ID] == trace_id])

print("their anchor under each rule:")
display(pd.concat([
    anch_f[anch_f[SCH_A["customer"]] == trace_id].assign(design="fixed"),
    anch_h[anch_h[SCH_A["customer"]] == trace_id].assign(design="hazard"),
]).set_index("design"))

### 1d. Percentile grids — the saved ranking artifact

Raw balances have brutal right tails and drift with the rate environment, so
each numeric is **ranked 0–100 within its own product type and vintage**. The
grid of rank edges is fitted **once** on training rows and saved to
`percentile_grids.json`; scoring **reloads it, never refits** — refitting would
make an entity's feature depend on whoever else is in that day's batch. A
grouping cell is only used when it held at least `min_cell_size` rows at fit
time; thinner cells fall back to the next level in `percentile_group`.

In [ ]:
grids = pre.fit_percentile_grids(acq, cfg_pre)
grid_path = os.path.join(cfg_pre["io"]["artifact_dir"], "percentile_grids.json")
with open(grid_path, "w") as fh:
    json.dump(grids, fh)
print("saved:", grid_path)

if grids:
    name = next(iter(grids))
    spec = cfg_pre["numeric_columns"][name]
    print("\ngrid structure for %r (fallback levels: %s):"
          % (name, spec["percentile_group"]))
    for lvl, cells in grids[name].items():
        sizes = [c["n"] for c in cells.values()]
        print("  level %s: %3d cells, cell sizes %d..%d"
              % (lvl, len(cells), min(sizes), max(sizes)))
    lbl, cell = next(iter(grids[name]["0"].items()))
    print("  e.g. cell %r (n=%d): p0=%.0f p50=%.0f p100=%.0f"
          % (lbl, cell["n"], cell["edges"][0], cell["edges"][50],
             cell["edges"][100]))

acq = pre.apply_percentiles(acq, grids, cfg_pre)
pct_cols = [c + "_pct" for c in grids]
print("\ntraced entity, raw value vs within-group rank:")
display(acq.loc[acq[ID] == trace_id,
                [DATE, COL["product_type"]] + list(grids) + pct_cols])

### 1e. `build_features` — one row per entity, leakage-filtered

For each kept entity, only rows **at or before the anchor** are visible
(`hist = grp[grp[date] <= anchor]`). The row then carries three blocks, all
config-driven:

- `agg_*` — full-history aggregates (counts, sums, tenure, per-family counts)
- `pos*_*` — the last `sequence.window` acquisitions, position 1 = the anchor
- `gap_*` — months **between** consecutive acquisitions (never anchor-forward:
  that distance is the label)

This is the pipeline's only hot spot — a deliberate readable per-entity loop.

In [ ]:
t0 = time.time()
feat_h = pre.build_features(acq, anch_h, cfg_pre)
feat_f = pre.build_features(acq, anch_f, cfg_pre)
print("built features: hazard-anchored %s | fixed-anchored %s | %.1fs"
      % (feat_h.shape, feat_f.shape, time.time() - t0))

row = feat_h[feat_h[CUST] == trace_id].T
row.columns = [trace_id]
print("\nthe traced entity's feature row, block by block:")
for block in ("agg_", "pos", "gap_", "n_acq", "bundle"):
    part = row[row.index.str.startswith(block)]
    if len(part):
        display(part)

In [ ]:
# --- 1f. attach labels: the same features, two dataset shapes --------
flat_df = pre.build_flat_dataset(feat_f, anch_f)
hazard_df = pre.build_hazard_dataset(feat_h, anch_h, cfg_pre)

LBL_FLAT = ctx.label_column("flat")      # e.g. "label"
LBL_HAZ = ctx.label_column("hazard")     # e.g. "event"

flat_path = os.path.join(cfg_pre["io"]["output_dir"], "flat_dataset.csv")
hazard_path = os.path.join(cfg_pre["io"]["output_dir"], "hazard_dataset.csv")
flat_df.to_csv(flat_path, index=False)
hazard_df.to_csv(hazard_path, index=False)

print("flat:   %6d rows = 1 per entity        | %4d events"
      % (len(flat_df), int(flat_df[LBL_FLAT].sum())))
print("hazard: %6d rows = 1 per entity-interval| %4d events"
      % (len(hazard_df), int(hazard_df[LBL_HAZ].sum())))

bk = SCH["hazard_bookkeeping"]
print("\nthe traced entity expanded into intervals at risk —")
print("the event lands only in the interval containing the outcome:")
display(hazard_df.loc[hazard_df[CUST] == trace_id, [CUST] + bk + [LBL_HAZ]])

In [ ]:
# --- 1g. exclusions: every dropped entity, with its reason ------------
EX = SCH["exclusions"]
excl = pd.concat([
    anch_f[~anch_f[KEEP]].assign(**{EX["design"]: "fixed"}),
    anch_h[~anch_h[KEEP]].assign(**{EX["design"]: "hazard"}),
])[[EX["design"], SCH_A["customer"], REASON]]
excl.to_csv(os.path.join(cfg_pre["io"]["output_dir"], "exclusions.csv"),
            index=False)
print("dropped entities by design and reason:")
display(excl.groupby([EX["design"], REASON]).size().rename("entities")
        .to_frame())

---
## Stage 2 — modelling (`nbo_data_modeling/model.py`)

`design: flat|hazard` in `model_config.yaml` switches everything; the notebook
uses whichever design your config selects. Three rules to keep in mind while
reading the outputs:

- **Folds split on the entity, never the row** (`StratifiedGroupKFold`) — in
  the hazard design one entity owns many rows, and splitting by row leaks them
  across the fold boundary.
- **Negative downsampling drops whole entities** and reweights the survivors
  by `1/fraction`, so probabilities stay near the true base rate.
- **Row count is not sample size** — the event count is what limits how many
  features the data supports.

In [ ]:
# The walkthrough's modelling config: same as the real one, but reading
# the walkthrough dataset and (by default) fewer CV repeats.
cfg_mdl = copy.deepcopy(ctx.mdl)
design = cfg_mdl["design"]
cfg_mdl["io"]["input_path"] = flat_path if design == "flat" else hazard_path
cfg_mdl["io"]["output_dir"] = os.path.join(WALK, "model_output")
cfg_mdl["cv"]["n_repeats"] = CV_REPEATS

GROUP = cfg_mdl["columns"]["group"]
LABEL = cfg_mdl["columns"]["label"]

dfm = mdl.load_data(cfg_mdl)     # applies the negative downsampling
numeric, categorical = mdl.split_column_types(dfm, cfg_mdl)

print("\ndesign=%s | %d rows | %d entities | %d row-events"
      % (design, len(dfm), dfm[GROUP].nunique(), int(dfm[LABEL].sum())))
print("feature typing: %d numeric, %d categorical" % (len(numeric),
                                                      len(categorical)))
print("  numeric[:6]:    ", numeric[:6])
print("  categorical[:6]:", categorical[:6])
print("\n_sample_weight distribution (retained negatives are up-weighted):")
display(dfm.groupby("_sample_weight").size().rename("rows").to_frame())

In [ ]:
# --- 2b. the pipeline: impute → encode → scale → fit, ONE object -----
# One sklearn object means cross-validation cannot leak: imputers and
# scalers are refit inside every training fold.
from sklearn import set_config
set_config(display="diagram")
pipe_demo = mdl.build_pipeline(numeric, categorical, cfg_mdl)
pipe_demo

In [ ]:
# --- 2c. cross-validation → out-of-fold scores -----------------------
if RUN_CV:
    t0 = time.time()
    preds, fold_models = mdl.run_cv(dfm, numeric, categorical, cfg_mdl)
    print("CV done in %.0fs (%d folds x %d repeats = %d fits)"
          % (time.time() - t0, cfg_mdl["cv"]["n_splits"],
             cfg_mdl["cv"]["n_repeats"],
             cfg_mdl["cv"]["n_splits"] * cfg_mdl["cv"]["n_repeats"]))
    print("\nrow-level out-of-fold predictions:")
    display(preds.head(6))
else:
    preds, fold_models = None, None
    print("RUN_CV=False — modelling and targeting sections are skipped.")

### 2d. From per-interval hazards to one score per entity

In the hazard design the model predicts a hazard per interval; those combine
into `P(convert by H) = 1 − ∏(1 − h_k)`, so both designs land on the same
customer-level footing and their metrics are directly comparable. Entities
censored before the horizon are **dropped from evaluation** — their outcome is
genuinely unknown, not a negative. The cell below shows the arithmetic for the
traced entity.

In [ ]:
if RUN_CV:
    cust_preds = mdl.to_customer_level(preds, cfg_mdl)
    print("customer-level scores: %d rows (%d entities x %d repeat(s))"
          % (len(cust_preds), cust_preds[cfg_mdl["columns"]["id"]].nunique(),
             CV_REPEATS))
    if design == "hazard":
        one = preds[(preds[GROUP] == trace_id) & (preds["repeat"] == 0)]
        if len(one):
            n_bins = (cfg_mdl["evaluation"]["hazard_horizon_months"]
                      // cfg_mdl["evaluation"]["hazard_bin_months"])
            inside = one[one["interval"].astype(int) <= n_bins]
            h = inside["score"].values
            print("\ntraced entity, repeat 0 — per-interval hazards:",
                  np.round(h, 4))
            print("P(convert by %dm) = 1 - prod(1-h) = %.4f"
                  % (cfg_mdl["evaluation"]["hazard_horizon_months"],
                     1 - np.prod(1 - h)))
        else:
            print("\n(traced entity was downsampled out of this run)")
    display(cust_preds.head(6))
else:
    print("skipped (RUN_CV=False)")

In [ ]:
# --- 2e. metrics: PR-AUC and lift first, ROC-AUC for reference -------
if RUN_CV:
    summary, per_repeat = mdl.evaluate(cust_preds, cfg_mdl)
    display(summary)
    prev = float(summary.loc[summary["metric"] == "prevalence", "mean"])
    print("NB: prevalence here is of the SAMPLED population "
          "(negative_customer_fraction=%.2f), so lift is measured against "
          "%.1f%%, not the true base rate."
          % (cfg_mdl.get("sampling", {}).get("negative_customer_fraction", 1.0),
             100 * prev))
else:
    print("skipped (RUN_CV=False)")

In [ ]:
# --- 2f. calibration: predicted vs observed, by score bin ------------
if RUN_CV:
    cal = mdl.calibration_table(cust_preds, cfg_mdl)
    display(cal)
    hi = float(cal["mean_predicted"].max())
    show(charts.line(cal["mean_predicted"].tolist(),
                     {"observed": cal["observed_rate"].tolist()},
                     xlabel="mean predicted", ylabel="observed rate",
                     reference=([0, hi], [0, hi], "perfect calibration"),
                     title="Calibration on the walkthrough run"))
else:
    print("skipped (RUN_CV=False)")

In [ ]:
# --- 2g. coefficients + the sign-consistency stability check ---------
# A coefficient whose sign flips across folds is noise, not a finding;
# below ~0.7 consistency treat it as unproven.
if RUN_CV:
    feats = mdl.feature_report(fold_models, dfm, numeric, categorical, cfg_mdl)

    def readable(name):
        for p in ("num__", "cat__", "remainder__"):
            if name.startswith(p):
                name = name[len(p):]
                break
        return (name[len("missingindicator_"):] + " (missing)"
                if name.startswith("missingindicator_") else name)

    view = feats.head(15).copy()
    if "feature" in view.columns:
        view["feature"] = view["feature"].map(readable)
    display(view)
else:
    print("skipped (RUN_CV=False)")

---
## Stage 3 — reporting components (`nbo_report/`)

The same functions power both `eda_report.html` (whole population) and
`model_report.html` (high-propensity subset) — sharing the *functions* rather
than the outputs is what makes the two reports comparable.

### 3a. Markov sequence profiling (`nbo_report/sequences.py`)

Generic over any event alphabet: `sequences.event_column` in
`report_config.yaml` names the column whose values become the events (here:
`product_family` — change the config and this section profiles something else).
Histories are cut at the **same anchors** preprocessing computed, so the
leakage rule holds here too. Lift below rides with a Wilson lower bound so a
3-of-4 cell cannot outrank a 300-of-1000 one.

In [ ]:
cfg_seq = ctx.cfg["sequences"]
print("event alphabet column:", cfg_seq["event_column"])

seqs, seq_labels = sequences.build_sequences(
    acq, anch_h, cfg_pre, cfg_seq["event_column"], SCH_A)
res = sequences.profile(seqs, seq_labels, cfg_seq)

print("%d entities, %d with the event, base rate %.2f%%"
      % (res["n_customers"], res["n_converters"], 100 * res["base_rate"]))
print("alphabet:", res["alphabet"])

order = cfg_seq["orders"][min(1, len(cfg_seq["orders"]) - 1)]
print("\nterminal %d-grams ranked by lift (min support %d):"
      % (order, cfg_seq["min_support"]))
display(res["ngrams"][order].head(10))

In [ ]:
# The transition DIFFERENCE is the matrix worth reading: red cells are
# moves the event-group makes more often than everyone else.
tr = res["transitions"]
if tr is not None and len(tr["rows"]):
    show(charts.heatmap(
        tr["difference"], tr["rows"], tr["cols"],
        mode="diverging", center=0.0, value_fmt="%+.2f",
        cbar_label="P(next) difference",
        title="Transition probability: %s minus %s"
              % (ctx.labels["actor_name"], ctx.labels["non_actor_name"])))
else:
    print("not enough transitions for a matrix at this data size")

In [ ]:
# The fully rendered report section, exactly as it appears in the HTML
# reports — same function, same inputs.
show_card(sequences.render(res, cfg_seq, charts, html, ctx.labels,
                           heading_note="Rendered inside the notebook via "
                                        "the same sequences.render() the "
                                        "reports use."))

### 3b. Feature profiling (`nbo_report/profiling.py`)

Three components:
- **Lineage** — the raw→engineered map is *rebuilt from the config*, then
  diffed against the file, so config/code drift surfaces as "unmapped".
- **Information Value** — one strength ranking that reads the same for numeric
  and categorical features. Above 0.5 on a rare event is flagged as a likely
  leak, not a discovery.
- **Anchor asymmetry** — how much of a feature's class separation disappears
  when the fixed-horizon cutoff is removed, i.e. how much is the anchor rule
  rather than behaviour.

In [ ]:
lineage = profiling.lineage_map(cfg_pre, list(flat_df.columns), "flat", SCH)
n_unmapped = int((lineage["block"] == "unmapped").sum())
n_missing = int((~lineage["in_dataset"]).sum())
print("lineage: %d columns explained | %d unmapped | %d configured-but-absent"
      % (len(lineage), n_unmapped, n_missing))
display(lineage.head(12))

In [ ]:
num_f, cat_f = profiling.split_feature_types(
    flat_df, ctx.bookkeeping_columns("flat"))
cfg_prof = ctx.cfg["profiling"]

iv = profiling.iv_ranking(flat_df, num_f + cat_f, LBL_FLAT,
                          bins=cfg_prof["iv_bins"],
                          min_bin_count=cfg_prof["min_bin_count"])
display(iv.head(12))
top_iv = iv.head(12)
show(charts.barh(top_iv["feature"], top_iv["iv"].fillna(0.0),
                 xlabel="Information Value", value_fmt="%.3f",
                 title="Strongest features by IV (walkthrough data)"))

In [ ]:
# Event rate by bin for the single strongest feature — edit `feature`
# to profile any other column.
feature = iv["feature"].iloc[0]
tab, base = profiling.event_rate_by_bin(flat_df, feature, LBL_FLAT,
                                        bins=cfg_prof["decile_bins"],
                                        min_bin_count=cfg_prof["min_bin_count"])
display(tab)
show(charts.diverging_barh(tab["bin"], tab["lift"], center=1.0,
                           xlabel="lift vs the %.2f%% base rate" % (100 * base),
                           value_fmt="%.2fx", title=feature))

In [ ]:
# Anchor asymmetry: features whose separation collapses once the
# fixed-horizon cutoff is removed. A big shrinkage means the anchor
# rule, not the entity's behaviour, is doing the separating.
asym = profiling.anchor_asymmetry(flat_df, hazard_df, num_f, SCH)
if asym is not None and len(asym):
    display(asym)
else:
    print("not computable on this data (no shared positive-separation features)")

### 3c. Targeting (`nbo_report/targeting.py`)

**The two scores are kept apart on purpose.** The *ranking* uses out-of-fold
scores — every entity scored by a model that never trained on them. The
*persisted artifact* is refit for scoring **new** entities and is never used to
rank the entities it trained on: doing so is the easiest way to publish a
target list that looks excellent and performs at chance.

In [ ]:
if RUN_CV:
    scores = targeting.customer_scores(cust_preds, cfg_mdl)
    dec = targeting.decile_table(scores, ctx.cfg["targeting"]["score_deciles"])
    display(dec)
    show(charts.diverging_barh(
        ["decile %d" % d for d in dec["decile"]], dec["lift"], center=1.0,
        xlabel="lift vs base rate", value_fmt="%.2fx",
        title="Lift by out-of-fold score decile"))
else:
    print("skipped (RUN_CV=False)")

In [ ]:
# --- refit + persist the deliverable, then explain the top entities --
if RUN_CV:
    art_path = os.path.join(WALK, "model_output", "model.joblib")
    frac = cfg_mdl.get("sampling", {}).get("negative_customer_fraction", 1.0)
    # For speed the walkthrough refits on the sampled frame; main.py
    # reloads the FULL population first (targeting.refit_on_full_population).
    # The bundle records which frame it got either way.
    pipe, bundle = targeting.refit_and_persist(
        mdl, dfm, numeric, categorical, cfg_mdl, art_path,
        downsampled=frac < 1.0)
    ok, n_iter = targeting.converged(pipe)
    print("persisted:", art_path)
    print("converged:", ok, "| iterations:", n_iter)
    if ok is False:
        print("  NB: saga hit its iteration cap — expected at walkthrough "
              "size. The ranking is fine, but do not read coefficients "
              "from a non-converged fit; the full-size runs converge.")
    print("bundle note:", bundle["note"][:90], "...")

    per_cust = dfm.drop_duplicates(subset=[GROUP]).set_index(GROUP)
    top_ids = [i for i in scores[cfg_mdl["columns"]["id"]].head(TOP_N_PREVIEW)
               if i in per_cust.index]
    reasons, kind = targeting.reason_codes(
        pipe, per_cust.loc[top_ids, numeric + categorical],
        numeric, categorical,
        top_k=ctx.cfg["targeting"]["reason_codes"],
        exclude_columns=SCH["hazard_bookkeeping"])
    view = scores[scores[cfg_mdl["columns"]["id"]].isin(top_ids)].copy()
    view["reason_codes"] = view[cfg_mdl["columns"]["id"]].map(
        dict(zip(top_ids, reasons)))
    print("\ntop %d entities with %s reason codes:" % (len(view), kind))
    display(view[[cfg_mdl["columns"]["id"], "score", "actual",
                  "reason_codes"]])
else:
    print("skipped (RUN_CV=False)")

In [ ]:
# --- segments inside the high-propensity population ------------------
if RUN_CV:
    cfg_tgt = ctx.cfg["targeting"]
    cut = max(cfg_tgt["segments"]["k"],
              int(round(len(scores) * cfg_tgt["high_propensity_fraction"])))
    high_ids = [i for i in scores[cfg_mdl["columns"]["id"]].head(cut)
                if i in per_cust.index]
    high = per_cust.loc[high_ids]
    seg = targeting.segment_population(
        high, numeric, cfg_tgt["segments"],
        scores=scores.set_index(cfg_mdl["columns"]["id"])["score"])
    if seg is not None:
        print("segments over the top %d entities "
              "(described vs the high-propensity average, not vs zero):"
              % len(high))
        display(seg["table"])
    else:
        print("too few high-propensity entities to segment at this size")
else:
    print("skipped (RUN_CV=False)")

---
## Where to go from here

- **Full reports** — everything above, assembled with generated insight
  callouts, model-risk notes and the campaign CSV:

  ```bash
  python eda.py     # → outputs/reports/eda_report.html
  python main.py    # → outputs/reports/model_report.html + next_best_customers.csv
  ```

  Add `--no-run` to re-render from artifacts already on disk.

- **Point at different data permanently** — edit
  `nbo_data_preprocessing/config.yaml` (column roles, families, numerics) and
  `report_config.yaml` (`schema:`, `labels:`, sequence/profiling/targeting
  knobs). No code changes.

- **This notebook's outputs** live under `outputs/walkthrough/` and are safe to
  delete at any time.

- **Before quoting numbers from a real run**: set
  `sampling.negative_customer_fraction: 1.0` in `model_config.yaml`, and never
  cite synthetic-run metrics as evidence of achievable lift — the generator
  injects its signal deliberately.